<a href="https://colab.research.google.com/github/GUIVICFISHER/RiskNeutralKernel2026/blob/main/risk_analysis_stock_erase_01%20REVIEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Tickers for Dot-Com era analysis
dot_com_era_tickers = {
    'Cisco': 'CSCO',
    'Intel': 'INTC',
    'Microsoft': 'MSFT',
    'Amazon': 'AMZN'
}

# Define the specific period for Dot-Com analysis (e.g., 1997 to 2003)
START_DATE = "1997-01-01"
END_DATE = "2003-12-31"

plt.figure(figsize=(18, 15))
plt.suptitle('Dot-Com Era: Stock Price vs. Debt-to-Equity Ratio (1997-2003)', y=1.02, fontsize=16)

plot_idx = 1
for company_name, ticker_symbol in dot_com_era_tickers.items():
    print(f"\nProcessing {company_name} ({ticker_symbol})...")
    try:
        # 1. Fetch historical daily prices
        price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
        if price_data.empty:
            print(f"No price data found for {company_name}. Skipping.")
            continue

        price_series = pd.Series(dtype=float) # Initialize an empty Series

        if 'Adj Close' in price_data.columns and not price_data['Adj Close'].empty:
            price_series = price_data['Adj Close'].squeeze()
        elif 'Close' in price_data.columns and not price_data['Close'].empty:
            price_series = price_data['Close'].squeeze()

        if price_series.empty: # Check if a valid series was extracted
            print(f"No valid price series (Adj Close or Close) found for {company_name}. Skipping.")
            continue

        # 2. Fetch quarterly balance sheets to calculate Debt-to-Equity
        ticker = yf.Ticker(ticker_symbol)
        balance_sheet_quarterly = ticker.quarterly_balance_sheet

        # Check if balance sheet data is available and has required fields
        if balance_sheet_quarterly.empty:
            print(f"No quarterly balance sheet data found for {company_name}. Skipping.")
            continue

        # Transpose to have dates as index
        balance_sheet_quarterly = balance_sheet_quarterly.T

        # Find relevant debt and equity keys (checking for common variations)
        total_debt_keys = ['Total Debt', 'Long Term Debt And Capital Lease Obligation', 'Long Term Debt', 'Current Debt And Capital Lease Obligation']
        equity_keys = ['Stockholders Equity', 'Total Equity Gross Minority Interest', 'Share Holder Equity']

        total_debt = pd.Series(dtype=float)
        for key in total_debt_keys:
            if key in balance_sheet_quarterly.columns:
                total_debt = balance_sheet_quarterly[key]
                break
        if total_debt.empty:
             # Attempt to sum common debt components if 'Total Debt' not found directly
            current_debt = balance_sheet_quarterly.get('Current Debt And Capital Lease Obligation', 0)
            long_term_debt = balance_sheet_quarterly.get('Long Term Debt And Capital Lease Obligation', 0)
            if isinstance(current_debt, pd.Series) and isinstance(long_term_debt, pd.Series):
                total_debt = current_debt.fillna(0) + long_term_debt.fillna(0)
            elif isinstance(current_debt, pd.Series): total_debt = current_debt
            elif isinstance(long_term_debt, pd.Series): total_debt = long_term_debt

        shareholder_equity = pd.Series(dtype=float)
        for key in equity_keys:
            if key in balance_sheet_quarterly.columns:
                shareholder_equity = balance_sheet_quarterly[key]
                break

        # Filter D/E data to the specified period and drop NaNs
        period_date_range = pd.to_datetime(pd.date_range(START_DATE, END_DATE))

        total_debt_filtered = total_debt.loc[total_debt.index.intersection(period_date_range)].dropna()
        shareholder_equity_filtered = shareholder_equity.loc[shareholder_equity.index.intersection(period_date_range)].dropna()

        if total_debt_filtered.empty or shareholder_equity_filtered.empty:
            print(f"Could not find sufficient non-NaN debt or equity data for {company_name} within the specified period. Skipping.")
            continue

        # Align indices of total_debt and shareholder_equity for division
        common_index = total_debt_filtered.index.intersection(shareholder_equity_filtered.index)
        if common_index.empty:
            print(f"No common dates with non-NaN debt and equity data for {company_name}. Skipping.")
            continue

        total_debt_aligned = total_debt_filtered.loc[common_index]
        shareholder_equity_aligned = shareholder_equity_filtered.loc[common_index]

        # Handle zero shareholder equity to avoid division by zero
        # Replace zero equity with NaN so it gets dropped or handled gracefully
        shareholder_equity_aligned_no_zero = shareholder_equity_aligned.replace(0, np.nan)

        if shareholder_equity_aligned_no_zero.dropna().empty:
            print(f"Shareholder Equity is zero or NaN for all available dates for {company_name}. Skipping.")
            continue

        # Calculate Debt-to-Equity ratio
        debt_to_equity = total_debt_aligned / shareholder_equity_aligned_no_zero
        debt_to_equity = debt_to_equity.replace([np.inf, -np.inf], np.nan).dropna() # drop any remaining inf/nan from division

        if debt_to_equity.empty:
            print(f"Calculated Debt-to-Equity is empty after cleaning for {company_name}. Skipping.")
            continue

        # 3. Align D/E (quarterly) with daily prices
        # Create a daily date range within the analysis period
        full_date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
        aligned_de = pd.Series(index=full_date_range, dtype=float)

        # Forward-fill quarterly D/E values to daily dates
        for q_date in sorted(debt_to_equity.index):
            # Find all daily dates from this quarter-end until the next quarter-end or end of period
            if q_date <= pd.to_datetime(END_DATE):
                start_fill_date = q_date
                next_q_dates = [d for d in debt_to_equity.index if d > q_date]
                end_fill_date = next_q_dates[0] if next_q_dates else pd.to_datetime(END_DATE)

                # Ensure end_fill_date is within the overall range and after start_fill_date
                end_fill_date = min(end_fill_date, pd.to_datetime(END_DATE))

                daily_dates_to_fill = pd.date_range(start=start_fill_date, end=end_fill_date, freq='D')
                # Make sure the dates are valid for the index
                daily_dates_to_fill = daily_dates_to_fill[daily_dates_to_fill.isin(aligned_de.index)]

                if not daily_dates_to_fill.empty:
                    aligned_de.loc[daily_dates_to_fill] = debt_to_equity.loc[q_date]

        # Fill any remaining NaNs at the beginning with the first available value
        aligned_de = aligned_de.ffill().loc[price_series.index] # Align to price_series index after ffill

        # 4. Plotting
        ax1 = plt.subplot(2, 2, plot_idx) # Price on left y-axis
        color = 'tab:blue'
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Stock Price', color=color)
        ax1.plot(price_series.index, price_series, color=color, label='Stock Price')
        ax1.tick_params(axis='y', labelcolor=color)
        ax1.set_title(f'{company_name} ({ticker_symbol})')
        ax1.grid(True)
        ax1.tick_params(axis='x', rotation=45)

        ax2 = ax1.twinx() # Debt-to-Equity on right y-axis
        color = 'tab:red'
        ax2.set_ylabel('Debt-to-Equity Ratio', color=color)
        ax2.plot(aligned_de.index, aligned_de, color=color, linestyle='--', label='Debt-to-Equity Ratio')
        ax2.tick_params(axis='y', labelcolor=color)
        ax2.set_ylim(bottom=0) # Ensure D/E starts from 0 for better visualization

        # Combine legends
        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines + lines2, labels + labels2, loc='upper left')

        plot_idx += 1

    except Exception as e:
        print(f"Error processing {company_name} ({ticker_symbol}): {e}")

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.show()

/tmp/ipykernel_18942/197087235.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_18942/197087235.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
[*********************100%***********************]  1 of 1 completed


Processing Cisco (CSCO)...
Could not find sufficient non-NaN debt or equity data for Cisco within the specified period. Skipping.

Processing Intel (INTC)...



/tmp/ipykernel_18942/197087235.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
[*********************100%***********************]  1 of 1 completed


Could not find sufficient non-NaN debt or equity data for Intel within the specified period. Skipping.

Processing Microsoft (MSFT)...
Could not find sufficient non-NaN debt or equity data for Microsoft within the specified period. Skipping.

Processing Amazon (AMZN)...


/tmp/ipykernel_18942/197087235.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
[*********************100%***********************]  1 of 1 completed


Could not find sufficient non-NaN debt or equity data for Amazon within the specified period. Skipping.


<Figure size 1800x1500 with 0 Axes>

### Comparison of Bubble Indicators for Current Era Stocks

To facilitate a direct comparison of the 'Bubble Indicator' across all current era stocks, we will plot their indicators on a single graph. This allows us to observe which stocks exhibit similar patterns, higher volatility, or more pronounced 'bubble' characteristics relative to each other over the same time period.

### 1. Financial Modeling Prep (FMP)

In [ ]:
# Removed: Installation for 'financialdatareader' failed and is not required for FMP data fetching via 'requests'.
# !pip install financialdatareader

In [ ]:
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
FMP_API_KEY = userdata.get('FMP_API_KEY')

# Note: The original code intended to use requests for robustness, which is still the case.
import requests

ticker_symbol = 'AAPL'

print(f"\nAttempting to fetch Income Statement for {ticker_symbol} using FMP...")

try:
    # FMP API endpoint for quarterly income statement
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker_symbol}?period=quarter&apikey={FMP_API_KEY}"
    response = requests.get(url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    income_statement_data = response.json()

    if income_statement_data:
        income_statement_df = pd.DataFrame(income_statement_data)
        # Convert 'date' column to datetime and set as index for better analysis
        income_statement_df['date'] = pd.to_datetime(income_statement_df['date'])
        income_statement_df = income_statement_df.set_index('date').sort_index()
        print(f"\nIncome Statement data from FMP for {ticker_symbol} (first 5 rows):")
        display(income_statement_df.head())
    else:
        print(f"No Income Statement data found for {ticker_symbol} via FMP. Check API key and ticker.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching Income Statement for {ticker_symbol} with FMP: {e}")
    print("Please ensure your FMP API key is valid and correctly entered in Colab secrets as 'FMP_API_KEY'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

SecretNotFoundError: Secret FMP_API_KEY does not exist.

### 2. Quandl (now Nasdaq Data Link)

**Note:** While Quandl (now Nasdaq Data Link) offers some free datasets, its primary value often comes from premium, curated datasets. The free datasets usually require some exploration on their website to find what's available without a paid subscription.

To use Quandl:

1.  **Get an API Key:** Sign up on the [Nasdaq Data Link website](https://data.nasdaq.com/sign-up) to get your API key.
2.  **Store it securely:** Add your API key to Colab's Secrets Manager with the name `NASDAQ_DATA_LINK_API_KEY`.
3.  **Install the library:** I'll provide code to install the `Quandl` Python library.
4.  **Fetch data:** We'll attempt to fetch a sample free dataset, like a key economic indicator (e.g., from the FRED database, which is often available through Quandl).


In [ ]:
# Install the Quandl library
!pip install Quandl

In [ ]:
import Quandl
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
NASDAQ_DATA_LINK_API_KEY = userdata.get('NASDAQ_DATA_LINK_API_KEY')

# Set the API key for Quandl
Quandl.ApiConfig.api_key = NASDAQ_DATA_LINK_API_KEY

print("\nAttempting to fetch a sample free dataset from Quandl (e.g., FRED/GDP)...")

try:
    # Try fetching a common free economic dataset, like US GDP from FRED
    # You might need to explore Quandl's website for other free codes.
    data = Quandl.get("FRED/GDP", authtoken=NASDAQ_DATA_LINK_API_KEY)

    if not data.empty:
        print(f"\nSample data from Quandl (FRED/GDP - first 5 rows):")
        display(data.head())
    else:
        print("No data found for FRED/GDP. Please check the dataset code and your API key.")
except Quandl.QuandlError as e:
    print(f"Error fetching data from Quandl: {e}")
    print("Please ensure your Nasdaq Data Link API key is valid and correctly entered in Colab secrets as 'NASDAQ_DATA_LINK_API_KEY'.")
    print("Also, verify that 'FRED/GDP' is still a publicly accessible dataset, or try another free dataset code from their website.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### 3. Polygon.io

Polygon.io offers a free tier for delayed stock, options, forex, and crypto data. It's known for its comprehensive API.

To use Polygon.io:

1.  **Get an API Key:** Sign up on the [Polygon.io website](https://polygon.io/dashboard/signup) to obtain your free API key.
2.  **Store it securely:** Add your API key to Colab's Secrets Manager with the name `POLYGON_API_KEY`.
3.  **Install the library:** I'll provide code to install the `polygon-api-client` Python library.
4.  **Fetch data:** We'll fetch historical daily prices for a sample ticker (e.g., 'AAPL').


In [ ]:
# Install the Polygon API client library
!pip install polygon-api-client

In [ ]:
from polygon import RESTClient
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
POLYGON_API_KEY = userdata.get('POLYGON_API_KEY')

# Initialize the RESTClient
client = RESTClient(POLYGON_API_KEY)

ticker_symbol = 'AAPL'
start_date = '2023-01-01'
end_date = '2023-03-01'

print(f"\nAttempting to fetch historical daily prices for {ticker_symbol} from Polygon.io...")

try:
    # Fetch aggregates (daily bars) for the ticker
    # Note: Free tier is for delayed data and might have rate limits or specific date range restrictions.
    aggs = []
    for a in client.list_aggs(ticker=ticker_symbol,
                             multiplier=1,
                             timespan='day',
                             from_=start_date,
                             to=end_date,
                             limit=50000): # Increased limit to ensure more data if available
        aggs.append(a)

    if aggs:
        # Convert to DataFrame
        df = pd.DataFrame(aggs)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df = df.set_index('timestamp').sort_index()
        print(f"\nHistorical daily prices from Polygon.io for {ticker_symbol} (first 5 rows):")
        display(df.head())
    else:
        print(f"No historical data found for {ticker_symbol} via Polygon.io. Check API key, ticker, or date range.")

except Exception as e:
    print(f"Error fetching data from Polygon.io for {ticker_symbol}: {e}")
    print("Please ensure your Polygon.io API key is valid and correctly entered in Colab secrets as 'POLYGON_API_KEY'.")
    print("Remember the free tier has limitations (e.g., delayed data, rate limits).")

### 4. IEX Cloud (IEX Exchange API)

IEX Cloud offers market data, fundamental data, and news, with a developer-friendly API and a free tier with usage limits.

To use IEX Cloud:

1.  **Get an API Key:** Sign up on the [IEX Cloud Console](https://iexcloud.io/console/signup) to obtain your 'pk_' (publishable) API key.
2.  **Store it securely:** Add your API key to Colab's Secrets Manager with the name `IEX_CLOUD_API_KEY`.
3.  **Install the library:** I'll provide code to install the `iexfinance` Python library.
4.  **Fetch data:** We'll fetch historical prices for a sample ticker (e.g., 'AAPL').


In [ ]:
# Install the iexfinance library
!pip install iexfinance

In [ ]:
from iexfinance.stocks import get_historical_data
from iexfinance.stocks import Stock
from datetime import datetime
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
IEX_CLOUD_API_KEY = userdata.get('IEX_CLOUD_API_KEY')

ticker_symbol = 'AAPL'
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 3, 1)

print(f"\nAttempting to fetch historical daily prices for {ticker_symbol} from IEX Cloud...")

try:
    # Fetch historical data
    df = get_historical_data(ticker_symbol, start=start_date, end=end_date, output_format='pandas', token=IEX_CLOUD_API_KEY)

    if not df.empty:
        print(f"\nHistorical daily prices from IEX Cloud for {ticker_symbol} (first 5 rows):")
        display(df.head())
    else:
        print(f"No historical data found for {ticker_symbol} via IEX Cloud. Check API key, ticker, or date range. Remember free tier limits.")

except Exception as e:
    print(f"Error fetching data from IEX Cloud for {ticker_symbol}: {e}")
    print("Please ensure your IEX Cloud API key (starting with 'pk_') is valid and correctly entered in Colab secrets as 'IEX_CLOUD_API_KEY'.")
    print("Also, verify your free tier usage limits haven't been exceeded.")

### 5. Twelve Data

Twelve Data provides real-time and historical market data, including stocks, forex, crypto, and indices. They offer a free API plan with limited requests.

To use Twelve Data:

1.  **Get an API Key:** Sign up on the [Twelve Data website](https://twelvedata.com/pricing) for the 'Free API' plan.
2.  **Store it securely:** Add your API key to Colab's Secrets Manager with the name `TWELVE_DATA_API_KEY`.
3.  **Install the library:** I'll provide code to install the `twelvedata` Python library.
4.  **Fetch data:** We'll fetch historical daily prices for a sample ticker (e.g., 'AAPL').


In [ ]:
# Install the twelvedata library
!pip install twelvedata

In [ ]:
# Install the twelvedata library
!pip install twelvedata

from twelvedata import TDClient
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
TWELVE_DATA_API_KEY = userdata.get('TWELVE_DATA_API_KEY')

# Initialize the client
tdc = TDClient(apikey=TWELVE_DATA_API_KEY)

ticker_symbol = 'AAPL'

print(f"\nAttempting to fetch historical daily prices for {ticker_symbol} from Twelve Data...")

try:
    # Fetch historical time series data
    # Free plan usually allows only up to 30 days of historical data and 8 requests per minute.
    ts = tdc.time_series(symbol=ticker_symbol, interval="1day", outputsize=30).as_pandas()

    if not ts.empty:
        print(f"\nHistorical daily prices from Twelve Data for {ticker_symbol} (first 5 rows):")
        display(ts.head())
    else:
        print(f"No historical data found for {ticker_symbol} via Twelve Data. Check API key, ticker, or limits.")

except Exception as e:
    print(f"Error fetching data from Twelve Data for {ticker_symbol}: {e}")
    print("Please ensure your Twelve Data API key is valid and correctly entered in Colab secrets as 'TWELVE_DATA_API_KEY'.")
    print("Remember the free plan has strict limits on historical data depth and request frequency.")

ModuleNotFoundError: No module named 'twelvedata'

### 6. Tiingo

Tiingo offers financial data APIs for historical data, end-of-day, and real-time. They provide a free tier with daily updates for end-of-day data.

To use Tiingo:

1.  **Get an API Key:** Sign up on the [Tiingo website](https://api.tiingo.com/) for a free account.
2.  **Store it securely:** Add your API key to Colab's Secrets Manager with the name `TIINGO_API_KEY`.
3.  **Install the library:** I'll provide code to install the `tiingo` Python library.
4.  **Fetch data:** We'll fetch historical daily prices for a sample ticker (e.g., 'AAPL').


In [ ]:
# Install the tiingo library
!pip install tiingo

In [ ]:
from tiingo import TiingoClient
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
TIINGO_API_KEY = userdata.get('TIINGO_API_KEY')

# Initialize the client
config = {
    'session': True, # Use persistent session
    'api_key': TIINGO_API_KEY
}
client = TiingoClient(config)

ticker_symbol = 'AAPL'
start_date = '2023-01-01'
end_date = '2023-03-01'

print(f"\nAttempting to fetch historical daily prices for {ticker_symbol} from Tiingo...")

try:
    # Fetch historical daily prices
    # Free tier usually has limits on historical data and request frequency.
    historical_prices = client.get_dataframe(
        ticker_symbol,
        frequency='daily',
        startDate=start_date,
        endDate=end_date
    )

    if not historical_prices.empty:
        print(f"\nHistorical daily prices from Tiingo for {ticker_symbol} (first 5 rows):")
        display(historical_prices.head())
    else:
        print(f"No historical data found for {ticker_symbol} via Tiingo. Check API key, ticker, or date range.")

except Exception as e:
    print(f"Error fetching data from Tiingo for {ticker_symbol}: {e}")
    print("Please ensure your Tiingo API key is valid and correctly entered in Colab secrets as 'TIINGO_API_KEY'.")
    print("Remember the free tier has limitations.")


In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Tickers for Dot-Com era analysis
dot_com_era_tickers = {
    'Cisco': 'CSCO',
    'Intel': 'INTC',
    'Microsoft': 'MSFT',
    'Amazon': 'AMZN'
}

# Define the specific period for Dot-Com analysis (e.g., 1997 to 2003)
START_DATE = "1997-01-01"
END_DATE = "2003-12-31"

plt.figure(figsize=(18, 15))
plt.suptitle('Dot-Com Era: Stock Price vs. Debt-to-Equity Ratio (1997-2003)', y=1.02, fontsize=16)

plot_idx = 1
for company_name, ticker_symbol in dot_com_era_tickers.items():
    print(f"\nProcessing {company_name} ({ticker_symbol})...")
    try:
        # 1. Fetch historical daily prices
        price_data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")
        if price_data.empty:
            print(f"No price data found for {company_name}. Skipping.")
            continue

        price_series = price_data['Adj Close'].squeeze()
        if price_series.empty:
            price_series = price_data['Close'].squeeze() # Fallback if Adj Close is not available
        if price_series.empty:
            print(f"No valid price series found for {company_name}. Skipping.")
            continue

        # 2. Fetch quarterly balance sheets to calculate Debt-to-Equity
        ticker = yf.Ticker(ticker_symbol)
        balance_sheet_quarterly = ticker.quarterly_balance_sheet

        # Check if balance sheet data is available and has required fields
        if balance_sheet_quarterly.empty:
            print(f"No quarterly balance sheet data found for {company_name}. Skipping.")
            continue

        # Transpose to have dates as index
        balance_sheet_quarterly = balance_sheet_quarterly.T

        # Find relevant debt and equity keys (checking for common variations)
        total_debt_keys = ['Total Debt', 'Long Term Debt And Capital Lease Obligation', 'Long Term Debt', 'Current Debt And Capital Lease Obligation']
        equity_keys = ['Stockholders Equity', 'Total Equity Gross Minority Interest', 'Share Holder Equity']

        total_debt = pd.Series(dtype=float)
        for key in total_debt_keys:
            if key in balance_sheet_quarterly.columns:
                total_debt = balance_sheet_quarterly[key]
                break
        if total_debt.empty:
             # Attempt to sum common debt components if 'Total Debt' not found directly
            current_debt = balance_sheet_quarterly.get('Current Debt And Capital Lease Obligation', 0)
            long_term_debt = balance_sheet_quarterly.get('Long Term Debt And Capital Lease Obligation', 0)
            if isinstance(current_debt, pd.Series) and isinstance(long_term_debt, pd.Series):
                total_debt = current_debt.fillna(0) + long_term_debt.fillna(0)
            elif isinstance(current_debt, pd.Series): total_debt = current_debt
            elif isinstance(long_term_debt, pd.Series): total_debt = long_term_debt

        shareholder_equity = pd.Series(dtype=float)
        for key in equity_keys:
            if key in balance_sheet_quarterly.columns:
                shareholder_equity = balance_sheet_quarterly[key]
                break

        if total_debt.empty or shareholder_equity.empty:
            print(f"Could not find sufficient debt or equity data for {company_name}. Skipping.")
            continue

        # Filter D/E data to the specified period
        total_debt = total_debt.loc[total_debt.index.intersection(pd.to_datetime(pd.date_range(START_DATE, END_DATE)))]
        shareholder_equity = shareholder_equity.loc[shareholder_equity.index.intersection(pd.to_datetime(pd.date_range(START_DATE, END_DATE)))]

        # Calculate Debt-to-Equity ratio
        debt_to_equity = total_debt / shareholder_equity
        debt_to_equity = debt_to_equity.replace([np.inf, -np.inf], np.nan).dropna()

        if debt_to_equity.empty:
            print(f"Calculated Debt-to-Equity is empty for {company_name}. Skipping.")
            continue

        # 3. Align D/E (quarterly) with daily prices
        # Create a daily date range within the analysis period
        full_date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
        aligned_de = pd.Series(index=full_date_range, dtype=float)

        # Forward-fill quarterly D/E values to daily dates
        for q_date in sorted(debt_to_equity.index):
            # Find all daily dates from this quarter-end until the next quarter-end or end of period
            if q_date <= pd.to_datetime(END_DATE):
                start_fill_date = q_date
                next_q_dates = [d for d in debt_to_equity.index if d > q_date]
                end_fill_date = next_q_dates[0] if next_q_dates else pd.to_datetime(END_DATE)

                # Ensure end_fill_date is within the overall range and after start_fill_date
                end_fill_date = min(end_fill_date, pd.to_datetime(END_DATE))

                daily_dates_to_fill = pd.date_range(start=start_fill_date, end=end_fill_date, freq='D')
                # Make sure the dates are valid for the index
                daily_dates_to_fill = daily_dates_to_fill[daily_dates_to_fill.isin(aligned_de.index)]

                if not daily_dates_to_fill.empty:
                    aligned_de.loc[daily_dates_to_fill] = debt_to_equity.loc[q_date]

        # Fill any remaining NaNs at the beginning with the first available value
        aligned_de = aligned_de.ffill().loc[price_series.index] # Align to price_series index after ffill

        # 4. Plotting
        ax1 = plt.subplot(2, 2, plot_idx) # Price on left y-axis
        color = 'tab:blue'
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Stock Price', color=color)
        ax1.plot(price_series.index, price_series, color=color, label='Stock Price')
        ax1.tick_params(axis='y', labelcolor=color)
        ax1.set_title(f'{company_name} ({ticker_symbol})')
        ax1.grid(True)
        ax1.tick_params(axis='x', rotation=45)

        ax2 = ax1.twinx() # Debt-to-Equity on right y-axis
        color = 'tab:red'
        ax2.set_ylabel('Debt-to-Equity Ratio', color=color)
        ax2.plot(aligned_de.index, aligned_de, color=color, linestyle='--', label='Debt-to-Equity Ratio')
        ax2.tick_params(axis='y', labelcolor=color)
        ax2.set_ylim(bottom=0) # Ensure D/E starts from 0 for better visualization

        # Combine legends
        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines + lines2, labels + labels2, loc='upper left')

        plot_idx += 1

    except Exception as e:
        print(f"Error processing {company_name} ({ticker_symbol}): {e}")

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.show()

### Comparison of Bubble Indicators for Dot-Com Era Stocks

To analyze the 'Bubble Indicator' for the Dot-Com era, we will use a selection of prominent stocks from that period and examine their performance around the time of the bubble (e.g., 1997-2003). Plotting these on a single graph will allow us to observe common trends, individual stock volatility, and the manifestation of 'bubble' characteristics during that historical period.

The 'Bubble Indicator' is calculated as: $$\text{Bubble Indicator} = \frac{\text{Close Price} - \text{50-day SMA}}{\text{20-day SMA}}$$

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Tickers for Dot-Com era analysis
dot_com_era_tickers = {
    'Cisco': 'CSCO',
    'Intel': 'INTC',
    'Microsoft': 'MSFT',
    'Amazon': 'AMZN'
}

# Define SMA periods for the Bubble Indicator
SMA_SHORT = 20
SMA_LONG = 50

bubble_indicators_dot_com_df = pd.DataFrame()

# Define the specific period for Dot-Com analysis (e.g., 1997 to 2003)
START_DATE = "1997-01-01"
END_DATE = "2003-12-31"

print(f"Fetching data for Dot-Com era stocks from {START_DATE} to {END_DATE}...")

for company_name, ticker_symbol in dot_com_era_tickers.items():
    try:
        # Fetch historical data for the specified Dot-Com period
        data = yf.download(ticker_symbol, start=START_DATE, end=END_DATE, interval="1d")

        if data.empty:
            print(f"No historical data found for {company_name} ({ticker_symbol}) in the specified Dot-Com era. Skipping comparison for this stock.")
            continue

        # Determine the correct price series to use for calculations
        price_series = None
        if 'Adj Close' in data.columns:
            price_series = data['Adj Close'].squeeze()
        elif 'Close' in data.columns:
            price_series = data['Close'].squeeze()

        if price_series is None or not isinstance(price_series, pd.Series):
            print(f"Error: Could not extract a single price series for {company_name} ({ticker_symbol}). Skipping comparison.")
            continue

        # Calculate SMAs using the selected price_series
        data[f'SMA_{SMA_SHORT}'] = price_series.rolling(window=SMA_SHORT).mean()
        data[f'SMA_{SMA_LONG}'] = price_series.rolling(window=SMA_LONG).mean()

        # Calculate Bubble Indicator
        data['Bubble_Indicator'] = (price_series - data[f'SMA_{SMA_LONG}']) / data[f'SMA_{SMA_SHORT}']

        # Add to the comparison DataFrame
        bubble_indicators_dot_com_df[company_name] = data['Bubble_Indicator']

    except Exception as e:
        print(f"Error processing {company_name} ({ticker_symbol}) for comparison: {e}")

# Plotting the combined Bubble Indicators for Dot-Com era stocks
if not bubble_indicators_dot_com_df.empty:
    plt.figure(figsize=(15, 8))
    for column in bubble_indicators_dot_com_df.columns:
        plt.plot(bubble_indicators_dot_com_df.index, bubble_indicators_dot_com_df[column], label=f'{column} Bubble Indicator', alpha=0.8)

    plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.title('Comparison of Bubble Indicators for Dot-Com Era Stocks')
    plt.xlabel('Date')
    plt.ylabel('Bubble Indicator Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No Bubble Indicator data available for Dot-Com era comparison.")

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Tickers for current era analysis
current_era_tickers = {
    'Redwire': 'RDW',
    'Avantium': 'AVTX',
    'Wave Life Sciences': 'WVE',
    'Xometry': 'XMTR'
}

# Define SMA periods for the Bubble Indicator
SMA_SHORT = 20
SMA_LONG = 50

bubble_indicators_df = pd.DataFrame()

for company_name, ticker_symbol in current_era_tickers.items():
    try:
        # Fetch historical data for a sufficiently long period (e.g., 2 years)
        data = yf.download(ticker_symbol, period="2y", interval="1d")

        if data.empty:
            print(f"No historical data found for {company_name} ({ticker_symbol}). Skipping comparison for this stock.")
            continue

        # Determine the correct price series to use for calculations
        price_series = None
        if 'Adj Close' in data.columns:
            price_series = data['Adj Close'].squeeze()
        elif 'Close' in data.columns:
            price_series = data['Close'].squeeze()

        if price_series is None or not isinstance(price_series, pd.Series):
            print(f"Error: Could not extract a single price series for {company_name} ({ticker_symbol}). Skipping comparison.")
            continue

        # Calculate SMAs using the selected price_series
        data[f'SMA_{SMA_SHORT}'] = price_series.rolling(window=SMA_SHORT).mean()
        data[f'SMA_{SMA_LONG}'] = price_series.rolling(window=SMA_LONG).mean()

        # Calculate Bubble Indicator
        data['Bubble_Indicator'] = (price_series - data[f'SMA_{SMA_LONG}']) / data[f'SMA_{SMA_SHORT}']

        # Add to the comparison DataFrame
        bubble_indicators_df[company_name] = data['Bubble_Indicator']

    except Exception as e:
        print(f"Error processing {company_name} ({ticker_symbol}) for comparison: {e}")

# Plotting the combined Bubble Indicators
if not bubble_indicators_df.empty:
    plt.figure(figsize=(15, 8))
    for column in bubble_indicators_df.columns:
        plt.plot(bubble_indicators_df.index, bubble_indicators_df[column], label=f'{column} Bubble Indicator', alpha=0.8)

    plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.title('Comparison of Bubble Indicators for Current Era Stocks')
    plt.xlabel('Date')
    plt.ylabel('Bubble Indicator Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No Bubble Indicator data available for comparison.")

### Price Trends and Bubble Indicator for Current Era Stocks

To analyze potential bubble characteristics, we will:
1.  **Fetch historical price data** for the current era stocks (Redwire, Avantium, Wave Life Sciences, Xometry).
2.  **Calculate Simple Moving Averages (SMAs):** Specifically, 20-day and 50-day SMAs.
3.  **Calculate the 'Bubble Indicator':**
    $$\text{Bubble Indicator} = \frac{\text{Close Price} - \text{50-day SMA}}{\text{20-day SMA}}$$
    This indicator quantifies how much the current price has diverged from its longer-term trend (50-day SMA), relative to its short-term volatility/momentum (20-day SMA).
4.  **Plot the price trends, SMAs, and the Bubble Indicator** for each stock.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tickers for current era analysis (excluding Velo3D due to data limitations)
current_era_tickers = {
    'Redwire': 'RDW',
    'Avantium': 'AVTX',
    'Wave Life Sciences': 'WVE',
    'Xometry': 'XMTR'
}

# Define SMA periods for the Bubble Indicator
SMA_SHORT = 20
SMA_LONG = 50

plt.figure(figsize=(18, 12))

for i, (company_name, ticker_symbol) in enumerate(current_era_tickers.items()):
    try:
        # Fetch historical data for a sufficiently long period (e.g., 2 years)
        data = yf.download(ticker_symbol, period="2y", interval="1d")

        if data.empty:
            print(f"No historical data found for {company_name} ({ticker_symbol}).")
            continue

        # Determine the correct price series to use for calculations
        # Prioritize 'Adj Close' as it's the most accurate for historical analysis,
        # otherwise use 'Close'. Explicitly use .squeeze() to ensure it's a Series.
        price_series = None
        if 'Adj Close' in data.columns:
            price_series = data['Adj Close'].squeeze()
        elif 'Close' in data.columns:
            price_series = data['Close'].squeeze()
        else:
            print(f"Error: Neither 'Adj Close' nor 'Close' column found for {company_name} ({ticker_symbol}). Skipping.")
            continue

        # If price_series is not a Series (e.g., if squeeze() failed because it was already a multi-column DataFrame)
        if not isinstance(price_series, pd.Series):
            print(f"Error: Could not extract a single price series for {company_name} ({ticker_symbol}). Skipping.")
            continue

        # Calculate SMAs using the selected price_series
        data[f'SMA_{SMA_SHORT}'] = price_series.rolling(window=SMA_SHORT).mean()
        data[f'SMA_{SMA_LONG}'] = price_series.rolling(window=SMA_LONG).mean()

        # Calculate Bubble Indicator
        # Ensure we don't divide by zero or NaN for SMA_SHORT
        data['Bubble_Indicator'] = (price_series - data[f'SMA_{SMA_LONG}']) / data[f'SMA_{SMA_SHORT}']

        # Plotting
        plt.subplot(len(current_era_tickers), 2, 2*i + 1) # Price and SMAs
        plt.plot(price_series, label='Close Price', alpha=0.7) # Use price_series here
        plt.plot(data[f'SMA_{SMA_SHORT}'], label=f'{SMA_SHORT}-day SMA', alpha=0.7)
        plt.plot(data[f'SMA_{SMA_LONG}'], label=f'{SMA_LONG}-day SMA', alpha=0.7)
        plt.title(f'{company_name} ({ticker_symbol}) Price Trend & SMAs')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True)

        plt.subplot(len(current_era_tickers), 2, 2*i + 2) # Bubble Indicator
        plt.plot(data['Bubble_Indicator'], label='Bubble Indicator', color='red', alpha=0.7)
        plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        plt.title(f'{company_name} ({ticker_symbol}) Bubble Indicator')
        plt.xlabel('Date')
        plt.ylabel('Indicator Value')
        plt.legend()
        plt.grid(True)

    except Exception as e:
        print(f"Error processing {company_name} ({ticker_symbol}): {e}")

plt.tight_layout()
plt.suptitle('Price Trends, SMAs, and Bubble Indicator for Current Era Stocks', y=1.02, fontsize=16)
plt.show()

Certainly! Here's a comprehensive explanation of the mathematical expressions and conceptual framework for each financial metric we've discussed. I will break them down by category, providing the formula and defining its components.

### I. Profitability Ratios
These ratios measure a company's ability to generate earnings relative to its revenue, assets, or equity.

**Net Interest Margin (NIM)**

*   **Formula:** $$\text{NIM} = \frac{\text{Interest Income} - \text{Interest Expense}}{\text{Average Earning Assets}}$$
*   **Components:**
    *   **Interest Income:** Revenue generated from interest-bearing assets (e.g., loans, investments).
    *   **Interest Expense:** Cost incurred from interest-bearing liabilities (e.g., deposits, borrowed funds).
    *   **Average Earning Assets:** The average value of assets that generate interest income (e.g., average loans, average investment securities).
*   **Concept:** This is primarily a banking metric. It indicates how successfully a bank is investing its funds (loans, investments) compared to the cost of funding those investments.

**Return on Assets (ROA)**

*   **Formula:** $$\text{ROA} = \frac{\text{Net Income}}{\text{Total Assets}}$$
*   **Components:**
    *   **Net Income:** The company's total earnings after all expenses, including taxes and interest, have been deducted.
    *   **Total Assets:** The sum of all assets owned by the company (e.g., cash, inventory, property, plant, equipment).
*   **Concept:** Measures how efficiently a company is using its assets to generate profit. A higher ROA generally means more asset efficiency.

**Return on Equity (ROE)**

*   **Formula:** $$\text{ROE} = \frac{\text{Net Income}}{\text{Shareholder Equity}}$$
*   **Components:**
    *   **Net Income:** As defined above.
    *   **Shareholder Equity:** The total amount of capital invested by shareholders, plus retained earnings.
*   **Concept:** Measures the rate of return on the ownership interest of the common stockholders. It indicates how much profit a company generates for each dollar of equity.

**Earnings Per Share (EPS)**

*   **Formula:** $$\text{EPS} = \frac{\text{Net Income} - \text{Preferred Dividends}}{\text{Average Outstanding Shares}}$$
*   **Components:**
    *   **Net Income:** As defined above.
    *   **Preferred Dividends:** Dividends paid to holders of preferred stock, which are subtracted as they are not available to common shareholders.
    *   **Average Outstanding Shares:** The average number of common shares held by investors during the period.
*   **Concept:** Represents the portion of a company's profit allocated to each outstanding share of common stock. It's a key indicator of profitability for individual shares.

**Gross Profit Margin**

*   **Formula:** $$\text{Gross Profit Margin} = \frac{\text{Revenue} - \text{Cost of Goods Sold}}{\text{Revenue}} \times 100\%$$$
*   **Components:**
    *   **Revenue (or Sales):** The total amount of money generated from the sale of goods or services.
    *   **Cost of Goods Sold (COGS):** The direct costs attributable to the production of the goods sold by a company.
*   **Concept:** Indicates the percentage of revenue remaining after subtracting the direct costs of producing goods or services. It shows the company's ability to control production costs.

**Operating Profit Margin**

*   **Formula:** $$\text{Operating Profit Margin} = \frac{\text{Operating Income}}{\text{Revenue}} \times 100\%$$$
*   **Components:**
    *   **Operating Income (or EBIT - Earnings Before Interest and Taxes):** Gross profit minus operating expenses (e.g., selling, general, and administrative expenses).
    *   **Revenue:** As defined above.
*   **Concept:** Measures the percentage of revenue that is left after covering all operating expenses. It reflects the efficiency of a company's core operations.

**Net Profit Margin**

*   **Formula:** $$\text{Net Profit Margin} = \frac{\text{Net Income}}{\text{Revenue}} \times 100\%$$$
*   **Components:**
    *   **Net Income:** As defined above.
    *   **Revenue:** As defined above.
*   **Concept:** Shows the percentage of revenue that translates into net income. It's a comprehensive measure of profitability, taking all costs and taxes into account.

### II. Liquidity Ratios
These ratios assess a company's ability to meet its short-term obligations.

**Current Ratio**

*   **Formula:** $$\text{Current Ratio} = \frac{\text{Current Assets}}{\text{Current Liabilities}}$$
*   **Components:**
    *   **Current Assets:** Assets that can be converted into cash within one year (e.g., cash, accounts receivable, inventory).
    *   **Current Liabilities:** Obligations due within one year (e.g., accounts payable, short-term debt).
*   **Concept:** Indicates a company's ability to pay off its short-term debts with its short-term assets. A ratio greater than 1 is generally preferred.

**Quick Ratio (Acid-Test Ratio)**

*   **Formula:** $$\text{Quick Ratio} = \frac{\text{Current Assets} - \text{Inventory}}{\text{Current Liabilities}}$$
*   **Components:**
    *   **Current Assets, Inventory, Current Liabilities:** As defined above.
*   **Concept:** A more conservative measure of liquidity than the current ratio, as it excludes inventory (which can be difficult to quickly convert to cash) from current assets.

**Cash Ratio**

*   **Formula:** $$\text{Cash Ratio} = \frac{\text{Cash & Cash Equivalents}}{\text{Current Liabilities}}$$
*   **Components:**
    *   **Cash & Cash Equivalents:** The most liquid assets a company holds (e.g., cash, short-term marketable securities).
    *   **Current Liabilities:** As defined above.
*   **Concept:** The most stringent liquidity ratio, measuring a company's ability to pay off its current liabilities using only its cash and cash equivalents.

### III. Solvency Ratios
These ratios evaluate a company's ability to meet its long-term debt obligations.

**Debt-to-Equity Ratio**

*   **Formula:** $$\text{Debt-to-Equity Ratio} = \frac{\text{Total Debt}}{\text{Shareholder Equity}}$$
*   **Components:**
    *   **Total Debt:** Includes both short-term and long-term liabilities.
    *   **Shareholder Equity:** As defined above.
*   **Concept:** Measures the proportion of a company's assets financed by debt versus equity. A higher ratio indicates higher financial leverage and risk.

**Debt-to-Asset Ratio**

*   **Formula:** $$\text{Debt-to-Asset Ratio} = \frac{\text{Total Debt}}{\text{Total Assets}}$$
*   **Components:**
    *   **Total Debt:** As defined above.
    *   **Total Assets:** As defined above.
*   **Concept:** Indicates the percentage of a company's assets that are financed by creditors. A higher ratio means more assets are funded by debt.

**Interest Coverage Ratio**

*   **Formula:** $$\text{Interest Coverage Ratio} = \frac{\text{EBIT}}{\text{Interest Expense}}$$
*   **Components:**
    *   **EBIT (Earnings Before Interest and Taxes):** Operating income, as defined above.
    *   **Interest Expense:** The cost a company incurs for borrowed funds.
*   **Concept:** Measures a company's ability to meet its interest obligations. A higher ratio suggests that the company can more easily pay its interest expenses.

### IV. Efficiency Ratios
These ratios evaluate how efficiently a company uses its assets and manages its liabilities.

**Asset Turnover Ratio**

*   **Formula:** $$\text{Asset Turnover Ratio} = \frac{\text{Revenue}}{\text{Total Assets}}$$
*   **Components:**
    *   **Revenue:** As defined above.
    *   **Total Assets:** As defined above.
*   **Concept:** Measures how effectively a company is using its assets to generate sales. A higher ratio indicates better asset utilization.

**Inventory Turnover**

*   **Formula:** $$\text{Inventory Turnover} = \frac{\text{Cost of Goods Sold}}{\text{Average Inventory}}$$
*   **Components:**
    *   **Cost of Goods Sold (COGS):** As defined above.
    *   **Average Inventory:** The average value of inventory over a period (e.g., (Beginning Inventory + Ending Inventory) / 2).
*   **Concept:** Measures how many times a company has sold and replaced its inventory during a period. A higher turnover generally indicates efficient inventory management.

**Days Sales Outstanding (DSO)**

*   **Formula:** $$\text{DSO} = \frac{\text{Accounts Receivable}}{\text{Total Sales}} \times \text{Number of Days in Period}}$$
*   **Components:**
    *   **Accounts Receivable:** Money owed to the company by customers for goods or services delivered on credit.
    *   **Total Sales:** The total credit sales over the period.
    *   **Number of Days in Period:** Typically 365 for a year or 90 for a quarter.
*   **Concept:** Measures the average number of days it takes for a company to collect its accounts receivable. A lower DSO is generally better, indicating quicker cash collection.

### V. Valuation Ratios
These ratios are used to determine the financial attractiveness of an investment.

**Price-to-Earnings (P/E) Ratio**

*   **Formula:** $$\text{P/E Ratio} = \frac{\text{Share Price}}{\text{Earnings Per Share (EPS)}}$$
*   **Components:**
    *   **Share Price:** The current market price of one share of the company's stock.
    *   **Earnings Per Share (EPS):** As defined above.
*   **Concept:** Compares a company's current share price to its per-share earnings. It indicates how much investors are willing to pay for each dollar of earnings. A high P/E can suggest growth expectations.

**Price-to-Book (P/B) Ratio**

*   **Formula:** $$\text{P/B Ratio} = \frac{\text{Share Price}}{\text{Book Value Per Share}}$$
*   **Components:**
    *   **Share Price:** As defined above.
    *   **Book Value Per Share:** (Shareholder Equity - Preferred Equity) / Total Outstanding Shares. Represents the value of equity if all assets were sold and liabilities paid.
*   **Concept:** Compares a company's market value to its book value. It indicates how investors value the company relative to its accounting value of equity.

**Dividend Yield**

*   **Formula:** $$\text{Dividend Yield} = \frac{\text{Annual Dividends Per Share}}{\text{Share Price}} \times 100\%$$$
*   **Components:**
    *   **Annual Dividends Per Share:** The total dividends paid out per share over a year.
    *   **Share Price:** As defined above.
*   **Concept:** Shows the percentage return an investor receives in dividends relative to the stock's current price. It's a measure of dividend income.

**Market Capitalization**

*   **Formula:** $$\text{Market Capitalization} = \text{Share Price} \times \text{Number of Outstanding Shares}$$
*   **Components:**
    *   **Share Price:** As defined above.
    *   **Number of Outstanding Shares:** The total number of a company's shares currently held by all its shareholders.
*   **Concept:** Represents the total dollar market value of a company's outstanding shares. It's often used to classify company size (small-cap, mid-cap, large-cap).

### VI. Banking Specific Metrics
These ratios are particularly relevant for analyzing the financial health and performance of banks.

**Loan-to-Deposit Ratio**

*   **Formula:** $$\text{Loan-to-Deposit Ratio} = \frac{\text{Total Loans}}{\text{Total Deposits}}$$
*   **Components:**
    *   **Total Loans:** The total value of all loans issued by the bank.
    *   **Total Deposits:** The total value of all customer deposits held by the bank.
*   **Concept:** Measures a bank's liquidity and how much of its depositor's money it has loaned out. A high ratio might indicate less liquidity.

**Tier 1 Capital Ratio**

*   **Formula:** $$\text{Tier 1 Capital Ratio} = \frac{\text{Tier 1 Capital}}{\text{Risk-Weighted Assets}}$$
*   **Components:**
    *   **Tier 1 Capital:** Core capital of a bank, including common equity and disclosed reserves, which is a measure of a bank's financial strength.
    *   **Risk-Weighted Assets (RWA):** A bank's assets weighted by their credit risk. For example, cash has zero risk weight, while a loan might have a higher risk weight.
*   **Concept:** A key measure of a bank's financial strength and its ability to absorb potential losses. Higher ratios indicate a stronger capital base.

**Cost-to-Income Ratio**

*   **Formula:** $$\text{Cost-to-Income Ratio} = \frac{\text{Operating Expenses}}{\text{Operating Income}}$$
*   **Components:**
    *   **Operating Expenses:** All non-interest expenses incurred by the bank (e.g., salaries, rent, technology costs).
    *   **Operating Income:** Income from core banking operations before taxes.
*   **Concept:** Measures a bank's operational efficiency. A lower ratio indicates that the bank is more efficient in managing its costs relative to its income.

**Non-Performing Loan (NPL) Ratio**

*   **Formula:** $$\text{NPL Ratio} = \frac{\text{Non-Performing Loans}}{\text{Total Loans}}$$
*   **Components:**
    *   **Non-Performing Loans:** Loans where the borrower has failed to make scheduled payments for a specified period (typically 90 days).
    *   **Total Loans:** The total value of all loans issued by the bank.
*   **Concept:** Indicates the quality of a bank's loan portfolio. A higher NPL ratio suggests a greater risk of loan defaults and potential financial instability for the bank.

This comprehensive breakdown covers the mathematical foundation and conceptual significance of each metric. Please let me know if you have any further questions on specific metrics or require more detail on their application!

In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd

tickers = {
    'Redwire': 'RDW',
    'Avantium': 'AVTX',
    'Wave Life Sciences': 'WVE',
    'Xometry': 'XMTR',
    'Velo3D': 'VLD'
}

# Dictionary to store financial info for each ticker
financial_data = {}

for company_name, ticker_symbol in tickers.items():
    print(f"\nFetching data for {company_name} ({ticker_symbol})...")
    try:
        ticker = yf.Ticker(ticker_symbol)
        # Get basic info, which often contains market cap, sector, industry, etc.
        info = ticker.info

        # Filter for relevant information that might indicate risk or key characteristics
        # This is a starting point; we can expand this based on what 'high risk' implies.
        relevant_info = {
            'symbol': info.get('symbol'),
            'longName': info.get('longName'),
            'sector': info.get('sector'),
            'industry': info.get('industry'),
            'marketCap': info.get('marketCap'),
            'beta': info.get('beta'), # Beta indicates volatility relative to the market
            'trailingPE': info.get('trailingPE'),
            'forwardPE': info.get('forwardPE'),
            'debtToEquity': info.get('debtToEquity'),
            'profitMargins': info.get('profitMargins'),
            'fullTimeEmployees': info.get('fullTimeEmployees'),
            'shortRatio': info.get('shortRatio'), # High short ratio can indicate high risk/bearish sentiment
            'totalCash': info.get('totalCash'),
            'totalDebt': info.get('totalDebt'),
            'country': info.get('country')
        }
        financial_data[company_name] = relevant_info
        print(f"Successfully fetched data for {company_name}.")
    except Exception as e:
        print(f"Error fetching data for {company_name} ({ticker_symbol}): {e}")

# Convert the dictionary of financial data to a DataFrame for better display
df_financial_data = pd.DataFrame.from_dict(financial_data, orient='index')

# Display the DataFrame
display(df_financial_data)

In [ ]:
import yfinance as yf
import pandas as pd

ticker_symbol_vld = 'VLD'
print(f"\nInvestigating missing data for Velo3D ({ticker_symbol_vld})...")

try:
    ticker_vld = yf.Ticker(ticker_symbol_vld)

    print("\n--- Attempting to fetch Balance Sheet ---")
    balance_sheet_vld = ticker_vld.balance_sheet
    if not balance_sheet_vld.empty:
        display(balance_sheet_vld)
    else:
        print("Balance Sheet data not found for Velo3D.")

    print("\n--- Attempting to fetch Income Statement ---")
    financials_vld = ticker_vld.financials
    if not financials_vld.empty:
        display(financials_vld)
    else:
        print("Income Statement data not found for Velo3D.")

    print("\n--- Attempting to fetch Key Statistics (if available) ---")
    # Sometimes key_stats is available even if info is sparse
    # Note: yfinance does not have a direct `key_stats` attribute that returns a DataFrame like balance_sheet or financials.
    # Most key stats are part of the `info` attribute, which we already tried.
    # Let's re-fetch and check the raw 'info' again.
    info_vld = ticker_vld.info
    if info_vld and len(info_vld) > 5: # Check if info dictionary has more than just the very basic fields
        print("Full info dictionary:")
        for k, v in info_vld.items():
            print(f"  {k}: {v}")
    else:
        print("Detailed 'info' data still largely unavailable or very sparse.")

except Exception as e:
    print(f"Error investigating Velo3D data: {e}")


### Accessing Velo3D's Official Filings and Investor Information

For the most accurate and comprehensive financial data for Velo3D (VLD), particularly the balance sheets and income statements that were missing from yfinance, the following are the primary and most reliable sources:

1.  **Velo3D Investor Relations Website:**
    *   Go to the official Velo3D website (velo3d.com).
    *   Navigate to their 'Investor Relations' or 'Investors' section. This is typically found in the footer or a dedicated menu.
    *   Look for sections like 'Financial Reports', 'SEC Filings', 'Quarterly Results', or 'Annual Reports'. Here you will find their 10-K (annual report), 10-Q (quarterly report), and other official disclosures.

2.  **SEC EDGAR Database:**
    *   Visit the U.S. Securities and Exchange Commission (SEC) EDGAR database at [www.sec.gov/edgar/searchedgar/companysearch](https://www.sec.gov/edgar/searchedgar/companysearch).
    *   In the search bar, type 'Velo3D' or its ticker symbol 'VLD'.
    *   You'll find a list of all their public filings. The most relevant for detailed financial statements are:
        *   **10-K:** Annual report, provides audited financial statements (Balance Sheet, Income Statement, Cash Flow Statement) for the full fiscal year.
        *   **10-Q:** Quarterly report, provides unaudited financial statements for each quarter.
        *   **8-K:** Current report, used to announce major events that shareholders should know about.
    *   You can download these documents, often in structured data formats or view them directly, to extract the missing financial details.

### Using Alpha Vantage to Obtain Financial Data

Alpha Vantage is a popular alternative for financial data, offering a free tier that can be useful for fetching income statements, balance sheets, and other fundamental data.

**Step 1: Obtain an Alpha Vantage API Key**

1.  Go to the [Alpha Vantage website](https://www.alphavantage.co/support/#api-key).
2.  Sign up for a free API key. It will be sent to your email address.

**Step 2: Store Your API Key Securely in Colab**

It's crucial to keep your API key secure. In Google Colab, you can use the Secrets Manager:

1.  Click the "🔑 Secrets" icon on the left-hand panel.
2.  Click "Add new secret".
3.  For 'Name', enter `ALPHA_VANTAGE_API_KEY`.
4.  For 'Value', paste your Alpha Vantage API key.
5.  Make sure "Notebook access" is toggled on.

**Step 3: Install the Alpha Vantage Python Library**

Let's install the necessary Python library.

In [ ]:
pip install alpha_vantage

Now, let's import the library and fetch data for Velo3D (VLD). We'll try to get its Income Statement and Balance Sheet to fill the missing information.

In [ ]:
from alpha_vantage.fundamentaldata import FundamentalData
import pandas as pd
from google.colab import userdata

# Get the API key from Colab secrets
API_KEY = userdata.get('ALPHA_VANTAGE_API_KEY')

# Initialize the FundamentalData client
fd = FundamentalData(key=API_KEY, output_format='pandas')

ticker_symbol_vld = 'VLD'

print(f"\nAttempting to fetch financial data for {ticker_symbol_vld} using Alpha Vantage...")

# --- Fetch Income Statement ---
try:
    income_statement_vld, meta_data_is = fd.get_income_statement_quarterly(symbol=ticker_symbol_vld)
    if not income_statement_vld.empty:
        print(f"\nIncome Statement data from Alpha Vantage for {ticker_symbol_vld}:")
        display(income_statement_vld.head())
        # Fill missing data in previous df_financial_data if needed
        # For now, just display that we got it
    else:
        print(f"No Income Statement data found for {ticker_symbol_vld} via Alpha Vantage.")
except Exception as e:
    print(f"Error fetching Income Statement for {ticker_symbol_vld} with Alpha Vantage: {e}")

# --- Fetch Balance Sheet ---
try:
    balance_sheet_vld_av, meta_data_bs = fd.get_balance_sheet_quarterly(symbol=ticker_symbol_vld)
    if not balance_sheet_vld_av.empty:
        print(f"\nBalance Sheet data from Alpha Vantage for {ticker_symbol_vld}:")
        display(balance_sheet_vld_av.head())
        # Fill missing data in previous df_financial_data if needed
        # For now, just display that we got it
    else:
        print(f"No Balance Sheet data found for {ticker_symbol_vld} via Alpha Vantage.")
except Exception as e:
    print(f"Error fetching Balance Sheet for {ticker_symbol_vld} with Alpha Vantage: {e}")

print("\n--- Attempting to update df_financial_data with Alpha Vantage data for VLD ---")
# This part would involve more complex data merging if specific fields are desired
# For demonstration, we just show we fetched the raw statements.
# To truly 'fill null data' in `df_financial_data`, we'd map Alpha Vantage fields
# to the columns of df_financial_data. This depends on what specific metrics
# we want from the balance sheet/income statement to populate in the summary.

# For now, we will simply note that we have fetched the statements, and they can be used for further analysis.

### Exploring Alternatives for Velo3D (VLD) Data

Given the limited data availability for Velo3D (VLD) through `yfinance`, here are some alternative strategies to obtain more comprehensive financial information:

1.  **Official Company Investor Relations Websites:**
    *   Publicly traded companies are required to disclose financial information. Check Velo3D's official investor relations website for their latest annual reports (10-K), quarterly reports (10-Q), and other financial filings. These documents (e.g., balance sheets, income statements, cash flow statements) are the most accurate and detailed source of information.

2.  **SEC Filings (EDGAR Database):**
    *   The U.S. Securities and Exchange Commission (SEC) EDGAR database is a public resource where all U.S. public companies file their financial reports. You can search for Velo3D (VLD) to access their 10-K, 10-Q, and other relevant documents directly. This is often the primary source for financial data platforms.

3.  **Other Financial Data Providers:**
    *   Consider using other financial data APIs or platforms that might have more extensive coverage, especially for smaller or newer public companies. Examples include Bloomberg Terminal (paid), Refinitiv Eikon (paid), FactSet (paid), Alpha Vantage (free tier available), Financial Modeling Prep (free tier available), or Quandl (some free data, some paid).

4.  **News and Financial Media Outlets:**
    *   Reputable financial news sources (e.g., Wall Street Journal, Bloomberg, Reuters, Seeking Alpha) often report on company financials and provide summaries or analysis. While not raw data, they can offer insights and point to official reports.

5.  **Direct Web Scraping (with caution):**
    *   If financial statements are presented in a structured way on the company's website or an investor portal, it might be possible to use web scraping libraries (like `BeautifulSoup` or `Selenium` in Python) to extract data. However, this method requires careful implementation, adherence to terms of service, and can be fragile if website structures change.

For `Velo3D (VLD)`, **checking their investor relations website or the SEC EDGAR database would be the most reliable and recommended first steps** to obtain their official financial statements.

In [ ]:
import yfinance as yf
import pandas as pd

# Re-using the tickers dictionary from previous cells
tickers = {
    'Redwire': 'RDW',
    'Avantium': 'AVTX',
    'Wave Life Sciences': 'WVE',
    'Xometry': 'XMTR',
    'Velo3D': 'VLD'
}

missing_data_summary = {}

print("\n--- Investigating missing data for all tickers ---")

for company_name, ticker_symbol in tickers.items():
    print(f"\nInvestigating {company_name} ({ticker_symbol})...")
    try:
        ticker = yf.Ticker(ticker_symbol)

        # Check Balance Sheet
        balance_sheet = ticker.balance_sheet
        if not balance_sheet.empty:
            print(f"  Balance Sheet data found for {company_name}. First 5 rows:\n{balance_sheet.head().to_string()}")
            missing_data_summary[company_name + ' Balance Sheet'] = 'Available'
        else:
            print(f"  Balance Sheet data not found for {company_name}.")
            missing_data_summary[company_name + ' Balance Sheet'] = 'Not Available'

        # Check Income Statement (financials)
        financials = ticker.financials
        if not financials.empty:
            print(f"  Income Statement data found for {company_name}. First 5 rows:\n{financials.head().to_string()}")
            missing_data_summary[company_name + ' Income Statement'] = 'Available'
        else:
            print(f"  Income Statement data not found for {company_name}.")
            missing_data_summary[company_name + ' Income Statement'] = 'Not Available'

        # Check detailed 'info' again, similar to the Velo3D specific check
        info = ticker.info
        if info and len(info) > 5 and info.get('marketCap') is not None: # Check for a substantial info dictionary and key metrics
            print(f"  Detailed 'info' data appears comprehensive for {company_name}.")
            missing_data_summary[company_name + ' Detailed Info'] = 'Comprehensive'
        else:
            print(f"  Detailed 'info' data is sparse or missing key metrics for {company_name}.")
            missing_data_summary[company_name + ' Detailed Info'] = 'Sparse/Missing Key Metrics'

    except Exception as e:
        print(f"  Error investigating data for {company_name}: {e}")
        missing_data_summary[company_name + ' General Error'] = str(e)

# Display a summary of missing data across all tickers
df_missing_summary = pd.DataFrame.from_dict(missing_data_summary, orient='index', columns=['Status'])
df_missing_summary.index.name = 'Data Point'
print("\n--- Summary of Data Availability ---")
display(df_missing_summary)

### Initial Analysis of Available Financial Data (Excluding Velo3D)

We will now proceed with an initial analysis of the financial data collected for Redwire (RDW), Avantium (AVTX), Wave Life Sciences (WVE), and Xometry (XMTR), as these have comprehensive information available through `yfinance`. Velo3D (VLD) is excluded from this immediate analysis due to its missing data, which we aim to resolve using Alpha Vantage.

Let's first display the filtered DataFrame containing only the companies with comprehensive data, and then perform some basic data exploration.

In [ ]:
# Filter out Velo3D from the financial_data DataFrame for this analysis
df_financial_data_filtered = df_financial_data[df_financial_data.index != 'Velo3D'].copy()

print("Financial Data for Companies with Comprehensive Information (Excluding Velo3D):")
display(df_financial_data_filtered)

print("\n--- Basic Data Exploration ---")
print("Data Types:")
display(df_financial_data_filtered.info())

print("\nDescriptive Statistics for Numerical Columns:")
display(df_financial_data_filtered.describe())

print("\nValue Counts for Categorical Columns (Sector, Industry):")
for col in ['sector', 'industry']:
    if col in df_financial_data_filtered.columns:
        print(f"\n{col.capitalize()}:")
        display(df_financial_data_filtered[col].value_counts())

This initial look gives us an overview of the available metrics. We can now consider specific financial ratios or comparisons for these companies. Since many metrics like `beta`, `marketCap`, `trailingPE`, `forwardPE`, `debtToEquity`, and `profitMargins` are already present, we can visualize these for comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics to visualize for comparison
metrics_to_plot = [
    'marketCap',
    'beta',
    'trailingPE',
    'forwardPE',
    'profitMargins',
    'debtToEquity'
]

plt.figure(figsize=(15, 10))
for i, metric in enumerate(metrics_to_plot):
    if metric in df_financial_data_filtered.columns:
        plt.subplot(2, 3, i + 1) # Arrange plots in a 2x3 grid
        sns.barplot(x=df_financial_data_filtered.index, y=metric, data=df_financial_data_filtered)
        plt.title(f'{metric} by Company')
        plt.ylabel(metric.replace('Cap', ' Capitalization').replace('PE', ' P/E Ratio').replace('Margins', ' Margins').replace('To', ' to '))
        plt.xlabel('') # Remove x-axis label as company names are on ticks
        plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.suptitle('Comparison of Key Financial Metrics (Excluding Velo3D)', y=1.02, fontsize=16) # Add a super title
plt.show()